# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202302_Earthquake_Turkiye'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'aria'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 12 .tif files in the S3 bucket.


['drcs_activations/202302_Earthquake_Turkiye/aria/ARIA_DPM_Sentinel-1_Turkiye_EQ.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_A014_20230209_20230221_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/20230

## Configure bucket and paths (no need to create session manually)

In [10]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [11]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 0
  - Total size: 0.00 GB


(0, 0)

In [12]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [13]:
keys

['drcs_activations/202302_Earthquake_Turkiye/aria/ARIA_DPM_Sentinel-1_Turkiye_EQ.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_A014_20230209_20230221_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/20230

In [15]:
# Define filename creator functions for different file types

def create_cog_filename_aria_month(f, EVENT_NAME):
    """Create COG filename for ARIA files, adding event name and YYYY-MM_monthly."""
    filename = Path(f).stem
    
    # Extract YYYYMM from EVENT_NAME (format: YYYYMM_EventType_Location)
    event_parts = EVENT_NAME.split('_')
    if event_parts and len(event_parts[0]) >= 6:
        year_month = event_parts[0][:6]  # 202302
        
        # Format as YYYY-MM
        year = year_month[:4]  # 2023
        month = year_month[4:6]  # 02
        formatted_date = f'{year}-{month}'
        
        # Create new filename: EVENT_NAME_original_filename_YYYY-MM_monthly.tif
        cog_filename = f'{EVENT_NAME}_{filename}_{formatted_date}_monthly.tif'
    else:
        # Fallback if EVENT_NAME doesn't have expected format
        cog_filename = f'{EVENT_NAME}_{filename}.tif'
    
    return cog_filename

filter_str = 'ARIA_DPM_Sentinel-1_Turkiye_EQ'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_aria_month(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202302_Earthquake_Turkiye_ARIA_DPM_Sentinel-1_Turkiye_EQ_2023-02_monthly.tif


In [16]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_aria_month, 
                                target_dir = "Sentinel-1/DPM", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202302_Earthquake_Turkiye_ARIA_DPM_Sentinel-1_Turkiye_EQ_2023-02_monthly.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202302_Earthquake_Turkiye/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/DPM

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202302_Earthquake_Turkiye

[1/1] Processing: drcs_activations/202302_Earthquake_Turkiye/aria/ARIA_DPM_Sentinel-1_Turkiye_EQ.tif
   Output filename: 202302_Earthquake_Turkiye_ARIA_DPM_Sentinel-1_Turkiye_EQ_2023-02_monthly.tif
   [MEMORY] Initial: 294.0 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=221285/1000000
            Estimated data coverage: 15.9% (from distributed samples)
   [VERIFY] Band 2: min=0, max=255, center sample non-zero=160

/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmp2rhz09f3_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp_y8y_l2w.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/DPM/202302_Earthquake_Turkiye_ARIA_DPM_Sentinel-1_Turkiye_EQ_2023-02_monthly.tif
   [MEMORY] Final: 1209.5 MB (Change: +915.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_ARIA_DPM_Sentinel-1_Turkiye_EQ_2023-02_monthly.tif

✅ Batch processing complete: 1 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/DPM/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/DPM/files_converted.csv
📁 COGs saved locally to: output/202302_Earthquake_Turkiye

📊 BATCH PROCESSING SUMMARY
Total files processed: 1
Successful: 1
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-19T14:54:44.156434


In [18]:
keys

['drcs_activations/202302_Earthquake_Turkiye/aria/ARIA_DPM_Sentinel-1_Turkiye_EQ.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_A014_20230209_20230221_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/20230

In [20]:
# Define filename creator functions for different file types
def create_cog_filename_date_range(f, EVENT_NAME):
    """Create COG filename moving date range to end, suffix before dates, and ensuring earlier date first."""
    from pathlib import Path
    
    # Get the subdirectory (aria or rgb)
    full_path = Path(f)
    parent_dir = full_path.parent.name  # Gets 'aria' or 'rgb'
    
    f2 = full_path.stem
    parts = f2.split('_')
    
    # Find date parts (YYYYMMDD format)
    date_indices = []
    dates_with_indices = []
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_indices.append(i)
            dates_with_indices.append((part, i))
    
    if len(dates_with_indices) == 2 and date_indices:
        # Extract just the dates
        dates = [d[0] for d in dates_with_indices]
        
        # Sort dates to ensure earlier date comes first
        dates.sort()
        
        # Format dates as YYYY-MM-DD
        formatted_date1 = f"{dates[0][:4]}-{dates[0][4:6]}-{dates[0][6:8]}"
        formatted_date2 = f"{dates[1][:4]}-{dates[1][4:6]}-{dates[1][6:8]}"
        
        # Get parts before first date occurrence
        first_date_index = min(date_indices)
        last_date_index = max(date_indices)
        
        # Get parts before first date
        prefix_parts = parts[:first_date_index]
        
        # Get parts between dates (if any) and after last date
        between_parts = []
        for i in range(first_date_index + 1, last_date_index):
            if i not in date_indices:
                between_parts.append(parts[i])
        
        # Get parts after last date
        suffix_parts = parts[last_date_index + 1:]
        
        # Reconstruct: EVENT_NAME + subdirectory + prefix + between + suffix + formatted dates
        all_parts = prefix_parts + between_parts + suffix_parts
        if all_parts:
            new_name = '_'.join(all_parts)
        else:
            new_name = ''
        
        # Build final filename with subdirectory and date format
        if new_name:
            cog_filename = f'{EVENT_NAME}_{parent_dir}_{new_name}_c{formatted_date1}_{formatted_date2}_day.tif'
        else:
            cog_filename = f'{EVENT_NAME}_{parent_dir}_c{formatted_date1}_{formatted_date2}_day.tif'
    else:
        # Fallback - still include subdirectory
        cog_filename = f'{EVENT_NAME}_{parent_dir}_{f2}.tif'
    
    return cog_filename

pattern = re.compile(r'.*aria.*Turkey.*_AZI\.tif$')


# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if pattern.match(i)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_date_range(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202302_Earthquake_Turkiye_aria_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
  202302_Earthquake_Turkiye_aria_Turkey_D021_AZI_c2023-01-29_2023-02-10_day.tif
  202302_Earthquake_Turkiye_aria_Turkey_S1_A014_AZI_c2023-01-28_2023-02-09_day.tif
  202302_Earthquake_Turkiye_rgb_Turkey_AZI_c2022-04-06_2023-02-08_day.tif


In [21]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_date_range, 
                                target_dir = "Sentinel-1/AZI", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202302_Earthquake_Turkiye_aria_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
  202302_Earthquake_Turkiye_aria_Turkey_D021_AZI_c2023-01-29_2023-02-10_day.tif
  202302_Earthquake_Turkiye_aria_Turkey_S1_A014_AZI_c2023-01-28_2023-02-09_day.tif
  202302_Earthquake_Turkiye_rgb_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202302_Earthquake_Turkiye/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/AZI

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202302_Earthquake_Turkiye

[1/4] Processing: drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_AZI.tif
   Output filename: 202302_Earthquake_Turkiye_aria_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
   [MEMORY] Initial: 1210.3 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checkin

Reading input: /tmp/tmp224kfdxo_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmppgmhs4ti.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/AZI/202302_Earthquake_Turkiye_aria_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
   [MEMORY] Final: 1225.2 MB (Change: +14.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_aria_Turkey_AZI_c2022-04-06_2023-02-08_day.tif

[2/4] Processing: drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_AZI.tif
   Output filename: 202302_Earthquake_Turkiye_aria_Turkey_D021_AZI_c2023-01-29_2023-02-10_day.tif
   [MEMORY] Initial: 1225.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-3.4028234663852886e+38, ma

...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/AZI/202302_Earthquake_Turkiye_aria_Turkey_S1_A014_AZI_c2023-01-28_2023-02-09_day.tif
   [MEMORY] Final: 1231.4 MB (Change: +4.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_aria_Turkey_S1_A014_AZI_c2023-01-28_2023-02-09_day.tif

[4/4] Processing: drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_AZI.tif
   Output filename: 202302_Earthquake_Turkiye_rgb_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
   [MEMORY] Initial: 1231.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=247, center 

/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmpl8jrzapz_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpszk09s_j.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/AZI/202302_Earthquake_Turkiye_rgb_Turkey_AZI_c2022-04-06_2023-02-08_day.tif
   [MEMORY] Final: 1231.5 MB (Change: +0.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_rgb_Turkey_AZI_c2022-04-06_2023-02-08_day.tif

✅ Batch processing complete: 4 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/AZI/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/AZI/files_converted.csv
📁 COGs saved locally to: output/202302_Earthquake_Turkiye

📊 BATCH PROCESSING SUMMARY
Total files processed: 4
Successful: 4
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-19T14:58:56.732176


In [22]:
keys

['drcs_activations/202302_Earthquake_Turkiye/aria/ARIA_DPM_Sentinel-1_Turkiye_EQ.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_20220406_20230208_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_A014_20230209_20230221_UNW.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_D021_20230129_20230210_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/Turkey_S1_A014_20230128_20230209_RNG.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_AZI.tif',
 'drcs_activations/202302_Earthquake_Turkiye/aria/rgb/Turkey_20220406_20230208_RNG.tif',
 'drcs_activations/20230

In [23]:
pattern = re.compile(r'^.*aria.*Turkey.*_RNG\.tif$')

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if pattern.match(i)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_date_range(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202302_Earthquake_Turkiye_aria_Turkey_RNG_c2022-04-06_2023-02-08_day.tif
  202302_Earthquake_Turkiye_aria_Turkey_D021_RNG_c2023-01-29_2023-02-10_day.tif
  202302_Earthquake_Turkiye_aria_Turkey_S1_A014_RNG_c2023-01-28_2023-02-09_day.tif
  202302_Earthquake_Turkiye_rgb_Turkey_RNG_c2022-04-06_2023-02-08_day.tif


In [ ]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_date_range, 
                                target_dir = "Sentinel-1/RNG", 
                                EVENT_NAME = EVENT_NAME)

In [ ]:
pattern = re.compile(r'.*aria.*Turkey.*_UNW\.tif$')

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if pattern.match(i)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_date_range(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


In [ ]:

# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_date_range, 
                                target_dir = "Sentinel-1/UNW", 
                                EVENT_NAME = EVENT_NAME)

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")